# Delta Analysis: Complexity Features Driving Quantum Advantage

Ridge regression with bootstrapping to identify which complexity features predict quantum vs classical performance differences.

## 1. Setup and Load Delta Analysis Data

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Try to import statsmodels, fall back to custom implementation
try:
    from statsmodels.stats.multitest import multipletests
    USE_STATSMODELS = True
    print("Using statsmodels for FDR correction")
except ImportError:
    USE_STATSMODELS = False
    print("statsmodels not available, using custom FDR implementation")

# Publication-quality settings
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

sns.set_style('whitegrid', {'grid.linestyle': '--', 'grid.alpha': 0.3})

# Load delta analysis CSV
PATH = '../../results/meta_analysis/'
delta_df = pd.read_csv(PATH + 'delta_analysis_best_methods_with_complexity.csv')

print(f"Loaded delta analysis: {delta_df.shape}")
print(f"Columns: {delta_df.columns.tolist()[:20]}...")

## 2. Data Cleaning - Remove High-NaN Columns

In [ ]:
# Calculate NaN percentage for each column
nan_percentages = (delta_df.isna().sum() / len(delta_df)) * 100

# Identify columns with >50% NaNs
high_nan_cols = nan_percentages[nan_percentages > 50].index.tolist()
print(f"\nColumns with >50% NaNs: {len(high_nan_cols)}")
if len(high_nan_cols) > 0:
    print("Dropping:", high_nan_cols[:10], "..." if len(high_nan_cols) > 10 else "")

# Drop high-NaN columns
delta_df_clean = delta_df.drop(columns=high_nan_cols)

# Identify complexity feature columns (those with _mean, _median, _std suffixes)
complexity_cols = [col for col in delta_df_clean.columns if 
                   col.endswith('_mean') or col.endswith('_median') or col.endswith('_std')]

print(f"\nAfter cleaning:")
print(f"  Total columns: {delta_df_clean.shape[1]}")
print(f"  Complexity features: {len(complexity_cols)}")
print(f"  Rows: {delta_df_clean.shape[0]}")

# Show remaining NaN percentages for complexity features
remaining_nans = (delta_df_clean[complexity_cols].isna().sum() / len(delta_df_clean)) * 100
print(f"\nRemaining NaN percentages in complexity features:")
print(f"  Max: {remaining_nans.max():.1f}%")
print(f"  Mean: {remaining_nans.mean():.1f}%")
print(f"  Features with >20% NaNs: {(remaining_nans > 20).sum()}")

## 3. FDR Correction Function

In [ ]:
def benjamini_hochberg_fdr(p_values, alpha=0.05):
    """Benjamini-Hochberg FDR correction (custom implementation)"""
    p_values = np.array(p_values)
    n = len(p_values)
    
    sorted_indices = np.argsort(p_values)
    sorted_p = p_values[sorted_indices]
    
    bh_critical = (np.arange(1, n + 1) / n) * alpha
    
    comparisons = sorted_p <= bh_critical
    if comparisons.any():
        max_i = np.where(comparisons)[0].max()
        reject_sorted = np.zeros(n, dtype=bool)
        reject_sorted[:max_i + 1] = True
    else:
        reject_sorted = np.zeros(n, dtype=bool)
    
    reject = np.zeros(n, dtype=bool)
    reject[sorted_indices] = reject_sorted
    
    p_corrected = np.zeros(n)
    p_corrected[sorted_indices] = np.minimum.accumulate(
        sorted_p * n / np.arange(1, n + 1)[::-1][::-1]
    )[::-1][::-1]
    p_corrected = np.minimum(p_corrected, 1.0)
    
    return reject, p_corrected

# Test
test_p = np.array([0.001, 0.01, 0.05, 0.1, 0.5])
reject, p_corr = benjamini_hochberg_fdr(test_p, alpha=0.05)
print("\nFDR correction test:")
print(f"  Original p-values: {test_p}")
print(f"  Corrected p-values: {p_corr}")
print(f"  Reject H0: {reject}")

## 4. Prepare Data for Ridge Regression

In [ ]:
def prepare_regression_data(df, task_name, complexity_features):
    """Prepare data for ridge regression for a specific task"""
    task_df = df[df['task'] == task_name].copy()
    
    y = task_df['percent_improvement_mean'].values
    X = task_df[complexity_features].copy()
    
    # Impute remaining NaNs with median
    for col in X.columns:
        if X[col].isna().any():
            X[col].fillna(X[col].median(), inplace=True)
    
    # Remove constant features
    valid_cols = []
    for col in X.columns:
        if X[col].std() > 0 and not X[col].isna().all():
            valid_cols.append(col)
    
    X = X[valid_cols]
    
    print(f"\n{task_name}:")
    print(f"  Samples: {len(y)}")
    print(f"  Features: {len(X.columns)}")
    print(f"  Target range: [{y.min():.2f}, {y.max():.2f}]")
    
    return X, y, valid_cols

# Prepare data for each task
tasks_data = {}
for task in ['Ranking', 'Classification', 'Link Prediction']:
    X, y, features = prepare_regression_data(delta_df_clean, task, complexity_cols)
    tasks_data[task] = {'X': X, 'y': y, 'features': features}

## 5. Ridge Regression with Bootstrapping

In [ ]:
def bootstrap_ridge_regression(X_df, y, feature_names, n_bootstrap=1000, alpha_range=np.logspace(-3, 3, 50)):
    """
    Perform ridge regression with bootstrapping
    FIXED: Pass feature_names separately to avoid AttributeError
    """
    n_samples, n_features = X_df.shape
    X = X_df.values  # Convert to numpy array
    
    # Standardize features
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Find optimal alpha using cross-validation
    ridge_cv = RidgeCV(alphas=alpha_range, cv=5)
    ridge_cv.fit(X_scaled, y)
    optimal_alpha = ridge_cv.alpha_
    
    print(f"  Optimal alpha: {optimal_alpha:.4f}")
    print(f"  CV R²: {ridge_cv.score(X_scaled, y):.4f}")
    
    # Bootstrap
    bootstrap_coefs = np.zeros((n_bootstrap, n_features))
    
    for i in range(n_bootstrap):
        indices = np.random.choice(n_samples, size=n_samples, replace=True)
        X_boot = X_scaled[indices]
        y_boot = y[indices]
        
        ridge = Ridge(alpha=optimal_alpha)
        ridge.fit(X_boot, y_boot)
        bootstrap_coefs[i] = ridge.coef_
    
    # Calculate statistics
    coefs_mean = bootstrap_coefs.mean(axis=0)
    coefs_std = bootstrap_coefs.std(axis=0)
    
    # Calculate p-values
    p_values = np.zeros(n_features)
    for j in range(n_features):
        if coefs_mean[j] > 0:
            p_values[j] = 2 * (bootstrap_coefs[:, j] < 0).mean()
        else:
            p_values[j] = 2 * (bootstrap_coefs[:, j] > 0).mean()
        p_values[j] = max(p_values[j], 1/n_bootstrap)
    
    return coefs_mean, coefs_std, p_values, feature_names, optimal_alpha

# Run ridge regression for each task
results = {}
output_dir = Path('../../results/meta_analysis/')

print("\n=== RIDGE REGRESSION WITH BOOTSTRAPPING ===")
for task_name, data in tasks_data.items():
    print(f"\n{task_name}:")
    coefs, coef_std, p_values, features, alpha = bootstrap_ridge_regression(
        data['X'], data['y'], data['features'], n_bootstrap=1000
    )
    
    # Apply FDR correction
    if USE_STATSMODELS:
        reject, p_values_fdr, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')
    else:
        reject, p_values_fdr = benjamini_hochberg_fdr(p_values, alpha=0.05)
    
    # Store results
    results[task_name] = {
        'coefficients': coefs,
        'std': coef_std,
        'p_values': p_values,
        'p_values_fdr': p_values_fdr,
        'significant_fdr': reject,
        'features': features,
        'alpha': alpha
    }
    
    print(f"  Significant features (FDR < 0.05): {reject.sum()}/{len(features)}")
    
    # Save detailed results to CSV
    results_df = pd.DataFrame({
        'feature': features,
        'coefficient': coefs,
        'std_error': coef_std,
        'p_value': p_values,
        'p_value_fdr': p_values_fdr,
        'significant_fdr': reject
    })
    results_df = results_df.sort_values('coefficient', key=abs, ascending=False)
    results_df.to_csv(output_dir / f'ridge_coefficients_{task_name.lower().replace(" ", "_")}.csv', index=False)
    
    print(f"  Top 5 positive coefficients:")
    top_pos = results_df[results_df['coefficient'] > 0].head(5)
    for _, row in top_pos.iterrows():
        sig = "***" if row['significant_fdr'] else ""
        print(f"    {row['feature']}: {row['coefficient']:.4f} ± {row['std_error']:.4f} {sig}")
    
    print(f"  Top 5 negative coefficients:")
    top_neg = results_df[results_df['coefficient'] < 0].head(5)
    for _, row in top_neg.iterrows():
        sig = "***" if row['significant_fdr'] else ""
        print(f"    {row['feature']}: {row['coefficient']:.4f} ± {row['std_error']:.4f} {sig}")

## 6. Forest Plots

In [ ]:
def create_forest_plot(task_name, coefficients, std_errors, p_values_fdr, features, 
                        significant, top_n=20, output_dir=None):
    """Create a forest plot showing top features with confidence intervals"""
    
    df = pd.DataFrame({
        'feature': features,
        'coef': coefficients,
        'std': std_errors,
        'p_fdr': p_values_fdr,
        'sig': significant
    })
    
    df['abs_coef'] = np.abs(df['coef'])
    df = df.sort_values('abs_coef', ascending=False).head(top_n)
    df = df.sort_values('coef')
    
    fig, ax = plt.subplots(figsize=(10, max(8, top_n * 0.4)))
    
    colors = ['#E74C3C' if sig else '#95A5A6' for sig in df['sig']]
    
    y_pos = np.arange(len(df))
    ax.errorbar(df['coef'], y_pos, xerr=1.96*df['std'], 
                fmt='o', markersize=8, capsize=5, capthick=2,
                color='none', ecolor=colors, elinewidth=2)
    ax.scatter(df['coef'], y_pos, c=colors, s=100, zorder=3, edgecolors='black', linewidths=1)
    
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, alpha=0.5)
    
    display_names = []
    for feat in df['feature']:
        name = feat.replace('_mean', '').replace('_median', '').replace('_std', '')
        name = name.replace('qbc_', '').replace('num_', '')
        if len(name) > 30:
            name = name[:27] + '...'
        display_names.append(name)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(display_names, fontsize=9)
    ax.set_xlabel('Standardized Coefficient', fontsize=12, fontweight='bold')
    ax.set_title(f'{task_name}\nTop {top_n} Features Predicting Quantum Advantage', 
                fontsize=13, fontweight='bold', pad=20)
    
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#E74C3C', label='Significant (FDR < 0.05)'),
        Patch(facecolor='#95A5A6', label='Not significant')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    
    ax.grid(True, axis='x', alpha=0.3, linestyle='--')
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    
    if output_dir:
        safe_name = task_name.lower().replace(' ', '_')
        plt.savefig(output_dir / f'forest_plot_{safe_name}.png', dpi=300, bbox_inches='tight')
        print(f"  Saved: forest_plot_{safe_name}.png")
    
    plt.show()
    return fig

# Create forest plots
print("\n=== CREATING FOREST PLOTS ===")
for task_name, res in results.items():
    print(f"\n{task_name}:")
    create_forest_plot(
        task_name,
        res['coefficients'],
        res['std'],
        res['p_values_fdr'],
        res['features'],
        res['significant_fdr'],
        top_n=20,
        output_dir=output_dir
    )

## 7. Summary Statistics

In [ ]:
print("\n=== SUMMARY STATISTICS ===\n")

for task_name, res in results.items():
    print(f"\n{task_name}:")
    print(f"  Ridge alpha: {res['alpha']:.4f}")
    print(f"  Total features: {len(res['features'])}")
    print(f"  Significant (FDR < 0.05): {res['significant_fdr'].sum()}")
    
    sig_mask = res['significant_fdr']
    if sig_mask.sum() > 0:
        sig_features = np.array(res['features'])[sig_mask]
        sig_coefs = res['coefficients'][sig_mask]
        
        print(f"\n  Significant features:")
        for feat, coef in sorted(zip(sig_features, sig_coefs), key=lambda x: abs(x[1]), reverse=True):
            direction = "↑" if coef > 0 else "↓"
            print(f"    {direction} {feat}: {coef:.4f}")
    else:
        print("  No features reached FDR significance threshold")

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("="*60)
print(f"\nGenerated files in {output_dir}:")
print("  - ridge_coefficients_ranking.csv")
print("  - ridge_coefficients_classification.csv")
print("  - ridge_coefficients_link_prediction.csv")
print("  - forest_plot_ranking.png")
print("  - forest_plot_classification.png")
print("  - forest_plot_link_prediction.png")

## Summary

Identified complexity features predicting quantum vs classical performance advantages.

### Methodology:
1. **Data Cleaning**: Removed columns with >50% NaNs
2. **Ridge Regression**: Cross-validated L2 regularization
3. **Bootstrapping**: 1000 samples for coefficient distributions
4. **P-values**: Two-tailed test (H0: coefficient = 0)
5. **FDR Correction**: Benjamini-Hochberg (α = 0.05)

### Statistical Rigor:
- Standardized features (mean=0, std=1)
- 5-fold cross-validation for alpha
- 95% bootstrap confidence intervals
- Multiple testing correction (FDR < 0.05)
- Supports both statsmodels and custom FDR

### Interpretation:
- **Positive coefficients**: Features favoring quantum methods
- **Negative coefficients**: Features favoring classical methods
- **Significance**: FDR < 0.05 threshold

### Output:
- CSV files with all coefficients and p-values
- Forest plots (top 20 features per task)
- Red = significant, Gray = not significant